# NeuroFetal AI — V5.0 Full Dataset Inference (Colab T4)

**Purpose:** Run the complete V5.0 Stacking Ensemble on the full CTU-UHB dataset
using a Colab T4 GPU (much faster than local CPU inference).

### Pipeline
| # | Phase | Description |
|---|-------|-------------|
| 1 | Setup | Clone repo, install deps, authenticate GitHub |
| 2 | Inference | Run `test_on_dataset.py` (all 15 models + meta-learner) |
| 3 | Results | View metrics, push report to GitHub |

### Models Loaded (15 total)
- **AttentionFusionResNet** × 5 folds (MC Dropout T=20)
- **1D-InceptionNet** × 5 folds
- **XGBoost** × 5 folds
- **Stacking Meta-Learner** (Logistic Regression)
- **Temperature Scaling** (T = 2.9088)

### Requirements
- **Google Colab with GPU** (T4 or better)
- **GitHub Personal Access Token (PAT)** stored in Colab Secrets as `GITHUB_TOKEN`

---

## How to Set Up GitHub Token in Colab Secrets

1. Click the **🔑 Key icon** in the left sidebar of Colab
2. Click **"+ Add new secret"**
3. Set **Name** = `GITHUB_TOKEN`
4. Set **Value** = your GitHub PAT (generate at [github.com/settings/tokens](https://github.com/settings/tokens))
5. Toggle the **"Notebook access"** switch ON
6. Done! The token will be available via `userdata.get('GITHUB_TOKEN')`

---
## 1. Setup Environment

### 1.1 Verify GPU Availability

In [ ]:
# Verify GPU is available
!nvidia-smi
import tensorflow as tf
print(f"\nTensorFlow: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

### 1.2 GitHub Authentication

In [ ]:
from google.colab import userdata
import os

# GitHub Authentication
GITHUB_REPO = "Krishna200608/NeuroFetal-AI"

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("✓ GitHub Token loaded from Secrets.")
except Exception as e:
    print("⚠️ Error loading GITHUB_TOKEN from Secrets. Falling back to manual input.")
    from getpass import getpass
    GITHUB_TOKEN = getpass("Enter GitHub Personal Access Token (PAT): ")

os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
os.environ['GITHUB_REPO'] = GITHUB_REPO

### 1.3 Clone Repository

In [ ]:
import shutil
import os

# Reset to /content
try:
    os.chdir("/content")
except:
    pass

# Clean up any previous clone
if os.path.exists("/content/NeuroFetal-AI"):
    shutil.rmtree("/content/NeuroFetal-AI")

print("Cloning repository...")
!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git

os.chdir("/content/NeuroFetal-AI")

# Stay on main branch (V5.0)
!git checkout main
!git pull origin main
print("✓ Cloned and checked out main!")

### 1.4 Git Credentials

In [ ]:
!git config --global user.email "krishnasikheriya001@gmail.com"
!git config --global user.name "Krishna200608"
print("✓ Git credentials set.")

### 1.5 Install Dependencies

In [ ]:
print("Installing libraries...")
!pip install -q wfdb scipy scikit-learn matplotlib seaborn pandas numpy \
    tensorflow xgboost lightgbm mne
print("✓ Dependencies installed.")

### 1.6 Verify Models Exist

In [ ]:
import glob
import os

model_dir = "/content/NeuroFetal-AI/Code/models"

resnet = sorted(glob.glob(os.path.join(model_dir, "enhanced_model_fold_*.keras")))
inception = sorted(glob.glob(os.path.join(model_dir, "inception_model_fold_*.keras")))
xgb_models = sorted(glob.glob(os.path.join(model_dir, "xgboost_model_fold_*.pkl")))
meta = os.path.exists(os.path.join(model_dir, "stacking_meta_learner.pkl"))
temp = os.path.exists(os.path.join(model_dir, "temperature_scaling.json"))

print(f"AttentionFusionResNet folds: {len(resnet)}")
print(f"InceptionNet folds:          {len(inception)}")
print(f"XGBoost folds:               {len(xgb_models)}")
print(f"Meta-Learner:                {'✓' if meta else '✗'}")
print(f"Temperature Scaling:         {'✓' if temp else '✗'}")

all_ok = len(resnet) >= 5 and len(inception) >= 5 and len(xgb_models) >= 5 and meta
print(f"\n{'✅ All 15 models + meta-learner found!' if all_ok else '⚠️ Some models missing.'}")

---
## 2. Run Full Dataset Inference

This runs the complete V5.0 pipeline:
1. Processes all 552 CTU-UHB records
2. Extracts 18 tabular + 19 CSP features per window
3. Runs **all 15 models** + stacking meta-learner
4. Applies **MC Dropout** (T=20) for uncertainty
5. Applies **Temperature Scaling** for calibration
6. Generates comprehensive Markdown + JSON reports

**Expected runtime:** ~3-5 minutes on T4 GPU

In [ ]:
!python Code/scripts/test_on_dataset.py

---
## 3. View Results

In [ ]:
# Display the generated Markdown report
from IPython.display import Markdown, display
import os

report_path = "/content/NeuroFetal-AI/Reports/Tests/final_metrics.md"

if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print("⚠️ Report not found. Check the inference output above for errors.")

In [ ]:
# View raw JSON results
import json

json_path = "/content/NeuroFetal-AI/Reports/Tests/final_metrics.json"

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        results = json.load(f)
    print(json.dumps(results, indent=2))
else:
    print("⚠️ JSON report not found.")

---
## 4. Push Results to GitHub

Commits and pushes the generated reports to the `main` branch.

In [ ]:
import os

os.chdir("/content/NeuroFetal-AI")

# Stage the test results
!git add Reports/Tests/final_metrics.md Reports/Tests/final_metrics.json

# Check if there are changes to commit
status = !git status --porcelain
if status:
    !git commit -m "results: V5.0 full dataset inference (Colab T4)"
    !git push origin main
    print("\n✅ Results pushed to GitHub successfully!")
else:
    print("ℹ️ No changes to commit (results already up to date).")

---

## Summary

After running this notebook, you will have:

1. ✅ **`Reports/Tests/final_metrics.md`** — Full Markdown report with AUC, Accuracy, Confusion Matrix, Uncertainty
2. ✅ **`Reports/Tests/final_metrics.json`** — Raw JSON for programmatic access
3. ✅ Results **pushed to GitHub** on the `main` branch

### Expected Performance (V5.0)
| Metric | Expected |
|---|---|
| AUC-ROC | ~0.86 |
| Accuracy | ~96% |
| Brier Score | ~0.046 |
| Model Size | 1.9 MB (Int8) |